# puc — run an experiment

Drives the flow end to end: **generate material → run episodes → read results**.

The run config (`configs/dev.toml`) is scenario-agnostic — the *what* (question + answers) comes from the generated corpus's manifest, passed in at run time.

In [ ]:
import json
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # ANTHROPIC_API_KEY from .env

SCENARIO = "scenarios/2_1.toml"      # generation input (the "what")
CONFIG = "configs/dev.toml"          # run config (the "how")
CORPUS = "generated_material/2_1/dev.md"
MODEL = "claude-sonnet-4-6"          # model used to generate the corpus

## 1. Generate material

Builds the background corpus the persuadee reads, from the scenario config. Volume knobs live in `[generation]` of the scenario TOML (kept small).

In [ ]:
from generate_material import generate

generate(SCENARIO, CORPUS, model=MODEL)

In [ ]:
print(Path(CORPUS).read_text())

## 2. Run episodes

Expands the run config against the corpus and runs each episode (actor answers; judge + monitor score the transcript). Writes one JSONL record per episode to `results/`.

In [ ]:
from run import run

out_path = run(CONFIG, CORPUS)
out_path

## 3. Read results

In [ ]:
records = [json.loads(line) for line in Path(out_path).read_text().splitlines()]

for rec in records:
    run_cfg = rec["run"]
    label = f"{run_cfg['condition']}" + (f"/{run_cfg['level']}" if run_cfg['level'] else "")
    print("=" * 80)
    print(label)
    if rec["error"]:
        print("ERROR:", rec["error"])
        continue
    for turn in rec["turns"]:
        who = "USER" if turn["speaker"] == "user" else "ACTOR"
        print(f"\n[{who}] {turn['text']}")
    print("\njudge  :", rec["judge_verdict"])
    print("monitor:", rec["monitor_verdict"])